In [2]:
%%writefile used-car-price-predictor/src/features.py
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

OWNER_RANK = {"First Owner": 1, "Second Owner": 2, "Third Owner": 3, "Fourth & Above Owner": 4}

NUM_FEATURES = ["car_age", "km_driven", "km_per_year", "engine_cc", "max_power_bhp",
                "power_per_litre", "mileage_kmpl", "torque_nm", "seats", "owner_rank"]
CAT_FEATURES = ["brand", "model", "fuel", "transmission", "seller_type"]
FEATURES = NUM_FEATURES + CAT_FEATURES


def brand_and_model(name):
    name = str(name).strip()
    brand = "Land Rover" if name.startswith("Land Rover") else name.split()[0]
    rest = name[len(brand):].strip()
    return brand, (rest.split()[0] if rest else "Unknown")


def add_features(df, ref_year):
    out = df.copy()
    out["car_age"] = (ref_year - out["year"]).clip(lower=0)
    out["km_per_year"] = out["km_driven"] / out["car_age"].clip(lower=1)
    out["power_per_litre"] = (out["max_power_bhp"] / out["engine_cc"] * 1000).replace([np.inf, -np.inf], np.nan)
    out["owner_rank"] = out["owner"].map(OWNER_RANK)
    return out


def build_preprocessor():
    numeric = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=15, sparse_output=False)
    return ColumnTransformer([("num", numeric, NUM_FEATURES), ("cat", categorical, CAT_FEATURES)])


def make_model(regressor):
    """Preprocessing + model in one pipeline; trains on log(price), predicts in rupees."""
    pipe = Pipeline([("prep", build_preprocessor()), ("model", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

Writing used-car-price-predictor/src/features.py


In [9]:
import sys, json

sys.path.append("used-car-price-predictor")

import numpy as np
import pandas as pd
from src.features import add_features, brand_and_model, NUM_FEATURES, CAT_FEATURES

df = pd.read_csv("used-car-price-predictor/data/processed/cars_clean.csv")
REF_YEAR = int(df["year"].max())
df[["brand", "model"]] = df["name"].apply(lambda n: pd.Series(brand_and_model(n)))
df = add_features(df, REF_YEAR)

json.dump({"ref_year": REF_YEAR}, open("used-car-price-predictor/models/meta.json", "w"))
print("Reference year:", REF_YEAR)
print("Brands:", df["brand"].nunique(), "| Models:", df["model"].nunique())
df[NUM_FEATURES + CAT_FEATURES].head()

Reference year: 2020
Brands: 32 | Models: 209


,car_age,km_driven,km_per_year,engine_cc,max_power_bhp,power_per_litre,mileage_kmpl,torque_nm,seats,owner_rank,brand,model,fuel,transmission,seller_type
0,6,145500,24250.000000,1248.0,74.00,59.294872,23.40,190.000000,5.0,1,Maruti,Swift,Diesel,Manual,Individual
1,6,120000,20000.000000,1498.0,103.52,69.105474,21.14,250.000000,5.0,2,Skoda,Rapid,Diesel,Manual,Individual
2,14,140000,10000.000000,1497.0,78.00,52.104208,17.70,124.544455,5.0,3,Honda,City,Petrol,Manual,Individual
3,10,127000,12700.000000,1396.0,90.00,64.469914,23.00,219.668960,5.0,1,Hyundai,i20,Diesel,Manual,Individual
4,13,120000,9230.769231,1298.0,88.20,67.950693,16.10,112.776475,5.0,1,Maruti,Swift,Petrol,Manual,Individual


In [10]:
print(df[NUM_FEATURES].isnull().sum())
df[["car_age", "km_per_year", "power_per_litre"]].describe().T

car_age              0
km_driven            0
km_per_year          0
engine_cc          208
max_power_bhp      209
power_per_litre    209
mileage_kmpl       223
torque_nm          211
seats              208
owner_rank           0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
car_age,6918.0,6.582105,4.076911,0.000000,3.000000,6.000000,9.000000,37.000000
km_per_year,6918.0,13216.246824,9334.494953,0.111111,7381.250000,11111.111111,16666.666667,120000.000000
power_per_litre,6709.0,62.519162,12.272231,21.312585,58.744993,64.249656,68.387636,203.148807


In [11]:
corr = df[NUM_FEATURES].corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Highly correlated pairs (>0.8):\n", pairs[pairs > 0.8])

print("\nSpearman correlation with log(price):")
print(df[NUM_FEATURES].corrwith(np.log1p(df["selling_price"]), method="spearman").sort_values(ascending=False).round(2))

Highly correlated pairs (>0.8):
 Series([], dtype: float64)

Spearman correlation with log(price):
max_power_bhp      0.63
torque_nm          0.60
engine_cc          0.48
km_per_year        0.38
power_per_litre    0.36
seats              0.32
mileage_kmpl       0.03
km_driven         -0.29
owner_rank        -0.40
car_age           -0.70
dtype: float64


In [13]:
from sklearn.model_selection import GroupShuffleSplit

key = ["name", "year", "km_driven", "fuel", "transmission", "owner"]
df["group_id"] = df.groupby(key).ngroup()
print("Rows sharing a vehicle key with another row:", df.duplicated(subset=key, keep=False).sum())

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["group_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

print("Train:", train.shape, "| Test:", test.shape)
print("Groups in both sets:", len(set(train["group_id"]) & set(test["group_id"])))
print(train["selling_price"].median(), test["selling_price"].median())

train.to_csv("used-car-price-predictor/data/processed/train.csv", index=False)
test.to_csv("used-car-price-predictor/data/processed/test.csv", index=False)

Rows sharing a vehicle key with another row: 453
Train: (5522, 20) | Test: (1396, 20)
Groups in both sets: 0
400000.0 425000.0
